# 7. Reshaping and Manipulating Arrays

Changing the shape, order of axes, or contents of an array.

In [1]:
import numpy as np

arr = np.arange(12).reshape(3, 4)
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

## Transposing

`arr.T`, `arr.transpose()` and `np.transpose(arr)` all do the same thing. They return a **view**, so no data is copied, and changes to the transpose show up in the original.

In [2]:
print(arr.T)
print(np.shares_memory(arr, arr.T))

[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
True


For arrays with more than 2 dimensions, you can give the new order of the axes. Remember that `transpose` doesn't modify the array, so you need to keep the result:

In [3]:
a = np.arange(24).reshape(2, 3, 4)
b = a.transpose(1, 0, 2)
print(a.shape, '->', b.shape)

(2, 3, 4) -> (3, 2, 4)


`swapaxes` swaps two axes, and `moveaxis` moves one axis to a new position. A common case is converting images between channels-last `(H, W, C)` and channels-first `(C, H, W)`:

In [4]:
img = np.zeros((32, 48, 3))     # height, width, channels
print(img.swapaxes(0, 2).shape)
print(np.moveaxis(img, -1, 0).shape)

(3, 48, 32)
(3, 32, 48)


## Reshaping

`reshape` returns a new array with the same data and a different shape. The total number of elements must stay the same. It returns a view when it can, and a copy when it has to.

In [5]:
print(arr.reshape(2, 6))
print(arr.reshape(-1))        # -1 means "work it out"
print(arr.reshape(4, -1).shape)

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
(4, 3)


In [6]:
try:
    arr.reshape(5, 3)
except ValueError as e:
    print('ValueError:', e)

ValueError: cannot reshape array of size 12 into shape (5,3)


By default elements are read and written row by row (`order='C'`). Pass `order='F'` to go column by column:

In [7]:
print(np.arange(6).reshape(2, 3))
print(np.arange(6).reshape(2, 3, order='F'))

[[0 1 2]
 [3 4 5]]
[[0 2 4]
 [1 3 5]]


`np.resize` is different: it doesn't require the size to match. It repeats the data to fill a bigger shape, or cuts it off for a smaller one.

(There's also the in-place method `arr.resize()`, which pads with zeros. It refuses to run if anything else references the array, and in Jupyter that's often the case, so the function is usually less hassle.)

In [8]:
print(np.resize(np.arange(4), (3, 3)))
print(np.resize(np.arange(10), (2, 3)))

[[0 1 2]
 [3 0 1]
 [2 3 0]]
[[0 1 2]
 [3 4 5]]


## Flattening

- `flatten()` always returns a **copy**
- `ravel()` returns a **view** if it can, otherwise a copy. It's faster, but changes may or may not affect the original, so don't rely on either behaviour.

In [9]:
f = arr.flatten()
r = arr.ravel()
print(np.shares_memory(arr, f), np.shares_memory(arr, r))

False True


In [10]:
# the transpose isn't contiguous, so ravel has to copy here
print(np.shares_memory(arr.T, arr.T.ravel()))

False


## Adding and removing dimensions

You often need an extra axis of length 1, for example to turn a 1-D vector into a column so it broadcasts correctly. `np.newaxis` (which is just `None`) and `np.expand_dims` both do this. `squeeze` removes length-1 axes.

In [11]:
v = np.array([1, 2, 3])
print(v.shape)
print(v[:, np.newaxis].shape)    # column
print(v[np.newaxis, :].shape)    # row
print(np.expand_dims(v, axis=0).shape)

(3,)
(3, 1)
(1, 3)
(1, 3)


In [12]:
x = np.zeros((1, 3, 1, 4))
print(x.squeeze().shape)
print(x.squeeze(axis=0).shape)

(3, 4)
(3, 1, 4)


## Flipping and rolling

In [13]:
print(np.flip(arr, axis=0))     # upside down
print(np.flip(arr, axis=1))     # left-right (same as arr[:, ::-1])

[[ 8  9 10 11]
 [ 4  5  6  7]
 [ 0  1  2  3]]
[[ 3  2  1  0]
 [ 7  6  5  4]
 [11 10  9  8]]


In [14]:
print(np.roll(np.arange(6), 2))
print(np.roll(arr, 1, axis=1))

[4 5 0 1 2 3]
[[ 3  0  1  2]
 [ 7  4  5  6]
 [11  8  9 10]]


In [15]:
np.rot90(arr)

array([[ 3,  7, 11],
       [ 2,  6, 10],
       [ 1,  5,  9],
       [ 0,  4,  8]])

## Adding and removing elements

`np.append`, `np.insert` and `np.delete` all return a **new array**. NumPy arrays have a fixed size, so there's no in-place append like with lists.

If you find yourself appending in a loop, collect the values in a Python list and convert once at the end. Appending to an array copies the whole thing every time.

In [16]:
a = np.array([1, 2, 3])
print(np.append(a, 4))
print(np.append(a, [4, 5, 6]))
print(a)    # unchanged

[1 2 3 4]
[1 2 3 4 5 6]
[1 2 3]


Without an `axis`, `append` flattens everything. Pass `axis` to add rows or columns:

In [17]:
m = np.array([[1, 2], [3, 4]])
print(np.append(m, [[5, 6]], axis=0))
print(np.append(m, [[9], [9]], axis=1))

[[1 2]
 [3 4]
 [5 6]]
[[1 2 9]
 [3 4 9]]


In [18]:
# np.insert(array, position, values)
print(np.insert(a, 0, 100))
print(np.insert(a, 1, [7, 8]))
print(np.insert(m, 1, 0, axis=1))   # insert a column of zeros

[100   1   2   3]
[1 7 8 2 3]
[[1 0 2]
 [3 0 4]]


In [19]:
print(np.delete(a, 0))
print(np.delete(a, [0, 2]))
print(np.delete(m, 0, axis=0))      # drop the first row

[2 3]
[2]
[[3 4]]


## Copying

Plain assignment doesn't copy anything, it just gives the same array a second name. Use `.copy()` when you need an independent array.

In [20]:
a = np.array([1, 2, 3])
b = a          # same object
c = a.copy()   # new data

a[0] = 99
print(b, c)

[99  2  3] [1 2 3]


`np.copy(a)` and `np.array(a, copy=True)` do the same thing as `a.copy()`.

One catch: for `object` arrays, `copy()` copies the references, not the objects themselves. If the elements are mutable (lists, dicts), use `copy.deepcopy`.